In [10]:
import os, re, glob
import numpy as np
import pandas as pd

# --- Paths ---
# CAM join-fit experiment output directory
CAM_JOIN_DIR = "/mnt/home/zwshi/learned-index/CAM/build/log/join_fit"

def extract_N(path: str) -> int | None:
    """Extract N (million queries) from filename: {dataset}_{N}Mquery_join.point.csv"""
    m = re.search(r'(\d+)Mquery', os.path.basename(path))
    return int(m.group(1)) if m else None


def detect_io_col(df):
    """Detect physical/cache-miss I/O count column."""
    if "IOs" in df.columns:
        return "IOs"
    if "avg_IOs" in df.columns:
        return "avg_IOs"
    raise KeyError("no I/O count column found (expected 'IOs' or 'avg_IOs')")


def detect_range_span_col(df):
    """Detect K, the number of pages scanned by a range probe."""
    if "range_pages" in df.columns:
        return "range_pages"
    if "DAC" in df.columns:
        return "DAC"
    if "logical_pages_read" in df.columns:
        return "logical_pages_read"
    return detect_io_col(df)


# --- Point fitting ---

def load_point_runs(glob_pattern, epsilon=16):
    """Load point-join experiment CSVs. Each file = one row with aggregate timing."""
    files = sorted(glob.glob(glob_pattern))
    if not files:
        print(f"[WARN] no files matching: {glob_pattern}")
        return pd.DataFrame()
    dfs = []
    for f in files:
        d = pd.read_csv(f)
        d = d[d["epsilon"] == epsilon].copy()
        if d.empty:
            continue
        d["file"] = os.path.basename(f)
        if "num_queries" in d.columns:
            d["N"] = d["num_queries"].astype(float)
        else:
            n = extract_N(f)
            d["N"] = float(n * 1e6) if n else np.nan
        dfs.append(d)
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()


def fit_point(data_dir=CAM_JOIN_DIR, epsilon=16):
    """
    Fit alpha (per-key CPU cost) and lambda_point (per-page I/O latency).

    Point CSV format (one row per N):
      epsilon, total_wall_time_s, IO_time_s, IOs, DAC,
      cache_hit_ratio, mem_time_s, IO_fraction, num_queries

    IOs = total I/O count summed across all N queries.
    total_wall_time_s = wall-clock time for the full run of N queries.
    """
    pattern = f"{data_dir}/*_*Mquery_join.point.csv"
    df = load_point_runs(pattern, epsilon=epsilon)
    if df.empty:
        print("[ERROR] no point data found")
        return

    IO_COL = detect_io_col(df)
    io_df = df[df[IO_COL] > 0].copy()
    if io_df.empty:
        print("[ERROR] no point rows with positive IOs")
        return
    io_df["lambda"] = io_df["IO_time_s"] / io_df[IO_COL]   # per-miss latency
    q1, q3 = io_df["lambda"].quantile([0.25, 0.75])
    iqr = q3 - q1
    df2 = io_df[(io_df["lambda"] >= q1 - 1.5 * iqr) & (io_df["lambda"] <= q3 + 1.5 * iqr)]
    if df2.empty:
        print("[ERROR] no point rows left after latency outlier filtering")
        return

    lambda_point = df2["lambda"].median()
    print("lambda_point (s/page)  =", lambda_point, "=", lambda_point * 1e6, "us/page")

    # df["T_cpu"] = df["total_wall_time_s"] - lambda_point * df[IO_COL]
    df["T_cpu"] = df["total_wall_time_s"] - df["IO_time_s"]
    alpha, delta = np.polyfit(df["N"].values, df["T_cpu"].values, 1)
    print("alpha     (s/key)       =", alpha)
    print("delta     (s, intercept)=", delta)
    return lambda_point, alpha, delta


# --- Range fitting ---

def load_range_runs(csv_path, epsilon=16):
    """Load range-join experiment CSV. One row per range query."""
    if not os.path.exists(csv_path):
        print(f"[WARN] file not found: {csv_path}")
        return pd.DataFrame()
    d = pd.read_csv(csv_path)
    d = d[d["epsilon"] == epsilon].copy()
    return d


def fit_range(data_dir=CAM_JOIN_DIR, epsilon=16):
    """
    Fit beta (per-page scan CPU), eta (fixed range overhead), lambda_range.

    Range CSV format (one row per generated range query):
      epsilon, query_idx, query_lo, query_hi, range_pages,
      total_wall_time_s, IO_time_s, IOs, DAC,
      cache_hit_ratio, mem_time_s, IO_fraction, num_queries, ...

    range_pages/DAC = page span K for the generated (lo, hi) query.
    IOs = cache misses for estimating lambda_range.
    """
    # new single-file format
    csv_path = f"{data_dir}/books_10M_uint64_unique_query_join.range.csv"
    df = load_range_runs(csv_path, epsilon=epsilon)

    # fall back to legacy multi-file glob
    if df.empty:
        files = sorted(glob.glob(f"{data_dir}/*query_join*.range.csv"))
        dfs = []
        for f in files:
            d = pd.read_csv(f)
            d = d[d["epsilon"] == epsilon]
            dfs.append(d)
        df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

    if df.empty:
        print("[ERROR] no range data found")
        return

    IO_COL = detect_io_col(df)
    K_COL = detect_range_span_col(df)
    df = df[df[K_COL] > 0].copy()
    if df.empty:
        print(f"[ERROR] no range rows with positive {K_COL}")
        return
    io_df = df[df[IO_COL] > 0].copy()
    if io_df.empty:
        print("[ERROR] no range rows with positive IOs")
        return
    io_df["lambda"] = io_df["IO_time_s"] / io_df[IO_COL]   # per-miss latency
    q1, q3 = io_df["lambda"].quantile([0.25, 0.75])
    iqr = q3 - q1
    df2 = io_df[(io_df["lambda"] >= q1 - 1.5 * iqr) & (io_df["lambda"] <= q3 + 1.5 * iqr)]
    if df2.empty:
        print("[ERROR] no range rows left after latency outlier filtering")
        return

    lambda_range = df2["lambda"].median()
    print("lambda_range (s/page)  =", lambda_range, "=", lambda_range * 1e6, "us/page")

    # K = page span; T_cpu = wall time minus measured physical/cache-miss I/O cost
    df["K"] = df[K_COL]
    # df["T_cpu"] = df["total_wall_time_s"] - lambda_range * df[IO_COL]
    df["T_cpu"] = df["total_wall_time_s"] - df["IO_time_s"]
    beta, eta = np.polyfit(df["K"].values, df["T_cpu"].values, 1)
    print("beta  (s/page_scan)     =", beta)
    print("eta   (s, fixed overhead)=", eta)
    return lambda_range, beta, eta


# --- Run ---
print("=== Fitting point cost model ===")
fit_point()

print("\n=== Fitting range cost model ===")
fit_range()


=== Fitting point cost model ===
lambda_point (s/page)  = 1.031707802134508e-06 = 1.0317078021345079 us/page
alpha     (s/key)       = 1.6229285547512437e-06
delta     (s, intercept)= 0.11490452489054716

=== Fitting range cost model ===
lambda_range (s/page)  = 3.56e-07 = 0.35600000000000004 us/page
beta  (s/page_scan)     = 1.7795920292443123e-06
eta   (s, fixed overhead)= 3.4811929872476325e-06


(np.float64(3.56e-07),
 np.float64(1.7795920292443123e-06),
 np.float64(3.4811929872476325e-06))